In [0]:
CATALOG = "workspace"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.study")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.study.raw")

study_notes = """
2026-10-22
Data Engineer in Python Track 30 min intro
"""

dbutils.fs.put(f"/Volumes/{CATALOG}/study/raw/study_notes.txt", study_notes.strip() + "\n", overwrite=True)

In [0]:
import csv, sys, os
from pathlib import Path

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.study_parser import parse_study_notes, parse_completions

CATALOG = "workspace"
text = Path(f"/Volumes/{CATALOG}/study/raw/study_notes.txt").read_text(encoding="utf-8")

rows, bad = parse_study_notes(text)
done = parse_completions(text)


def write_csv(path, header, data):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(data)


write_csv(f"/Volumes/{CATALOG}/study/raw/study_sessions.csv",
          ["date", "entry_no", "topic", "minutes", "note"], rows)
write_csv(f"/Volumes/{CATALOG}/study/raw/study_completions.csv",
          ["end_date", "course"], done)

print(f"parsed sessions: {len(rows)}, completions: {len(done)}")
for b in bad:
    print("SKIPPED:", b)

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"

for name, file in [("bronze_sessions", "study_sessions.csv"),
                   ("bronze_completions", "study_completions.csv")]:
    (spark.read.option("header", True).csv(f"/Volumes/{CATALOG}/study/raw/{file}")
        .withColumn("_ingested_at", F.current_timestamp())
        .write.format("delta").mode("overwrite")
        .saveAsTable(f"{CATALOG}.study.{name}"))

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.study.silver_sessions (
  study_date DATE, entry_no INT, topic STRING, minutes INT, note STRING
);

MERGE INTO workspace.study.silver_sessions AS t
USING (
  SELECT
    to_date(`date`) AS study_date,
    CAST(entry_no AS INT) AS entry_no,
    trim(topic) AS topic,
    CAST(minutes AS INT) AS minutes,
    note
  FROM workspace.study.bronze_sessions
  WHERE CAST(minutes AS INT) > 0
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY to_date(`date`), CAST(entry_no AS INT)
    ORDER BY _ingested_at DESC
  ) = 1
) AS s
ON t.study_date = s.study_date AND t.entry_no = s.entry_no
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.study.gold_weekly_hours AS
SELECT
  CAST(date_trunc('week', study_date) AS DATE) AS week_start,
  topic,
  ROUND(SUM(minutes) / 60.0, 2) AS hours,
  COUNT(*) AS sessions
FROM workspace.study.silver_sessions
GROUP BY CAST(date_trunc('week', study_date) AS DATE), topic

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.study.silver_completions (
  course STRING, end_date DATE
);

MERGE INTO workspace.study.silver_completions AS t
USING (
  SELECT trim(course) AS course, to_date(end_date) AS end_date
  FROM workspace.study.bronze_completions
  QUALIFY ROW_NUMBER() OVER (PARTITION BY trim(course) ORDER BY _ingested_at DESC) = 1
) AS s
ON t.course = s.course
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.study.course_plan AS
SELECT * FROM VALUES
  (1, 'Data Engineer in Python Track', 40.0),
  (2, 'Associate Data Engineer in SQL Track', 30.0),
  (3, 'Associate Data Engineer in Databricks Track', 37.0)
AS t(course_order, course, planned_hours)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.study.gold_courses AS
WITH actual AS (
  SELECT topic AS course,
         MIN(study_date) AS start_date,
         SUM(minutes) / 60.0 AS hours_done
  FROM workspace.study.silver_sessions
  GROUP BY topic
)
SELECT
  p.course_order,
  p.course,
  CAST(date_trunc('week', a.start_date) AS DATE) AS week_start,
  a.start_date,
  c.end_date,
  p.planned_hours,
  ROUND(COALESCE(a.hours_done, 0), 2) AS hours_done,
  ROUND(100 * COALESCE(a.hours_done, 0) / p.planned_hours, 1) AS progress_pct,
  CASE
    WHEN c.end_date IS NOT NULL THEN 'completed'
    WHEN a.start_date IS NOT NULL THEN 'in progress'
    ELSE 'not started'
  END AS status
FROM workspace.study.course_plan p
LEFT JOIN actual a ON a.course = p.course
LEFT JOIN workspace.study.silver_completions c ON c.course = p.course

In [0]:
%sql
SELECT * FROM workspace.study.gold_courses ORDER BY course_order